### RAG 란?
- LLM 에게 필요한 자료를 주고 답변하게 시키는 것
- LLM 은 똑똑한데, 모르는 것은 지어내고, 최신 정보는 없고, 사내 비밀 문서 이런것은 없다.
- 관련 질문을 했을 경우, 검색으로 근거를 찾아서 필요 자료를 줘야 하는 상황이다.

- 환각 감소
- 최신성 : 최신 문서를 검색에 넣어서 질문하게 시킬 수 있다.
- 출처 : 어떤 문서를 근거로 했는지 밝힐 수 있다.

- 챗봇, 문서 검색, 상당 시스템 등의 표준 구조

### RAG 흐름


In [2]:
import pandas as pd
df = pd.read_csv("../data/11-1_뉴스정제.csv").head(50)
df.head()


,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...
3,오늘부터 유류세 인하 폭 확대…하반기 바뀌는 세제·금융 정책은,img tag s 지난 30일 서울의 한 주유소. 〈사진 연합뉴스〉 img tag ...,경제,정부는 고유가 상황에 따라 국민의 유류비 부담 완화를 위해 이날부터 유류세를 법정 ...,https://n.news.naver.com/mnews/article/437/000...,img tag s 지난 30일 서울의 한 주유소 사진 연합뉴스 img tag e 오...
4,푸르덴셜생명 더 큰 드림 변액연금보험Ⅱ에 신규펀드 13종 추가,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...,경제,지난르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림 변액연금보...,https://n.news.naver.com/mnews/article/014/000...,파이낸셜뉴스 푸르덴셜생명보험은 급변하는 금융시장에 대응하기 위해 무배당 더 큰 드림...


In [4]:
# 가장 간단한 대화 - USER 만 쓰
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()       # api 
client = OpenAI()

In [3]:
docs = df['정제본문'].tolist()
docs[:5]


['서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복합몰을 만든다고 6일 밝혔다',
 '전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 30일 전북 전주시 전주교도소에서 보석으로 석방돼 기자들의 질의에 답변하고 있다 2022 06 30 pmkeul newsis com 서울 뉴시스 박정규 기자 이스타항공이 지난 30일 출소한 이상직 전 국회의원에 대해 이스타항공과 전혀 무관한 관계 라고 강조하면서 오해를 야기할 수 있는 언동을 하지 말 것을 경고했다 이스타항공은 3일 설명자료를 내고 현재까지도 이스타항공이 이 전 의원과 관계 있다고 오해될 여지가 있어 전혀 무관함을 분명히 하고자 한다 며 이같이 밝혔다 앞서 이 전 의원은 법원의 보석 허가로 전주교도소에서 출소하는 과정에서 취재진들에게 이스타항공이 좋은 회사가 되게끔 하겠다 며 해고된 직원들이 다시 취업하도록 돕겠다는 취지의 발언을 한 바 있다 이에 대해 이스타항공은 단순히 부적절한 정도를 넘어 새롭게 탈바꿈을 하고 재운항을 준비하고 있는 이스타항공의 진정성 있는 노력에 대내외적 불신을 야기할 수 있는 매우 심각한 문제 라며 향후 이스타항공과 관련이 있는 것으로 오해가 될 수 있는 어떠한 언동도 금해주시기를 요청드린다 고 밝혔다 또 다시 이러한 일이 발생할 경우 재발방지를 위한 모든 조치를 강구할 것임을 분명히 밝힌다 고 덧붙였다 이스타항공은 서울회생법원으로부터 인가된 회생계획에 따라 기존 최대주주인 이스타홀딩스 보유주식을 포함한 구주 전체가 소각됐다 면서 이 전 의원 측은 서울회생법원의 회생절차에서 어떠한 관여도 할 수 없었으며 회생계획에 따른 구주 전체의 무상소각 이후 이스타항공의 주식을 단 1주도 보유하고 있지 않은 이스타항공과 전혀 무관한 관계 라고 선을 그었다 아울러 이스타항공을 인수한 주식회사 성정 또한 이 전 의원과 전혀 관계가 없으며 특히 형남순 회장을 비롯한 관계인 그 누구도 이 전 의원과 일면식조차 없다 고 강조했다

In [11]:
import numpy as np

# 임베딩 함수 만들기
def embed(texts):
    """텍스트 목록을 받아서 벡터 배열로 변환하는 함수. openai embedding 을 사용해서 X 1536 차원으로 변화"""

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )

    return np.array([item.embedding for item in response.data])


In [12]:
doc_vecs = embed(docs)
doc_vecs[:5]

array([[-0.02064514, -0.00288582,  0.03421021, ...,  0.00049639,
        -0.02096558, -0.00830078],
       [-0.00181293,  0.01895142,  0.0059166 , ...,  0.00718689,
         0.03387451,  0.03408813],
       [ 0.03695679,  0.03442383,  0.00862122, ..., -0.02409363,
        -0.00515747, -0.01156616],
       [-0.00511169,  0.03097534,  0.00502777, ...,  0.01646423,
        -0.0375061 , -0.02575684],
       [ 0.01567078, -0.0001353 , -0.0093689 , ...,  0.02262878,
        -0.00443649,  0.01140594]], shape=(5, 1536))

### 1. 단계 - 질문으로 관련문서 검색하기
- 질문을 임베딩해야 합니다.

In [7]:
question = "리벨리온이라는 회사는 어떤 회사야?"


# 테스트용으로 근거없이 질문해서 답변 받아 봅시다.
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {'role':'user','content':question}

    ]
)


print(response.choices[0].message.content)


리벨리온(Rebellions)은 **AI 반도체를 설계하는 한국의 팹리스 기업**입니다. 직접 공장을 운영하기보다는 AI 연산에 특화된 칩을 설계하고, 생산은 삼성전자 같은 파운드리에 맡기는 방식입니다.

### 주요 사업
- **AI 가속기·NPU 개발**
  - 생성형 AI와 머신러닝 추론을 빠르고 전력 효율적으로 처리하는 반도체
  - 엔비디아 GPU를 일부 대체하거나 보완하는 것을 목표로 함
- **데이터센터용 AI 칩**
  - 대규모 언어 모델, 컴퓨터 비전, 클라우드 서비스 등에 사용
  - 특히 AI 모델을 실제 서비스에 적용하는 **추론(inference)** 분야에 초점
- **AI 소프트웨어 생태계**
  - 칩이 다양한 AI 프레임워크와 모델에서 작동하도록 컴파일러와 개발 도구도 함께 개발

### 대표 제품
- **ATOM(아톰)**: 데이터센터와 엣지 환경을 겨냥한 AI 추론용 NPU
- 이후 데이터센터용 성능과 확장성을 강화한 차세대 제품군도 개발하고 있습니다.

### 회사의 특징
- 2020년 설립된 비교적 젊은 회사
- 창업자는 삼성전자와 인텔 등에서 반도체 설계 경험을 쌓은 인력들
- 국내 AI 반도체 스타트업 가운데 기술력과 투자 유치 측면에서 가장 주목받는 기업 중 하나
- SK텔레콤 등 통신·클라우드 기업과 협력해 AI 데이터센터 시장 진출을 추진
- 2024년에는 **사피온코리아와의 통합**을 통해 국내 AI 반도체 역량을 결집하려는 움직임도 있었습니다.

한마디로 말하면, **“엔비디아 GPU 중심의 AI 반도체 시장에서 국내 기술로 데이터센터용 AI 칩을 만들려는 회사”**입니다. 다만 아직은 엔비디아처럼 세계적인 대규모 매출과 생태계를 갖춘 단계라기보다는, 기술과 제품을 상용화하며 시장을 확대하는 성장 기업에 가깝습니다.


In [13]:
query_vec = embed([question])[0] # embed 함수는 리스트로 입력을 받는다. 
query_vec

array([ 0.04122925, -0.03643799,  0.02145386, ...,  0.01843262,
       -0.01194   ,  0.01235962], shape=(1536,))

In [14]:
similarity = doc_vecs @ query_vec # 유사도 계산
similarity

array([0.09059599, 0.22485317, 0.20531869, 0.09310987, 0.1961786 ,
       0.20140506, 0.07952026, 0.06839526, 0.15579433, 0.33701467,
       0.23635968, 0.19862186, 0.14350459, 0.23317709, 0.24635898,
       0.26866426, 0.16338683, 0.22927509, 0.24102414, 0.28224229,
       0.14546497, 0.14204613, 0.23335706, 0.14291646, 0.18489471,
       0.07043638, 0.17242081, 0.2178767 , 0.205448  , 0.18155415,
       0.13164347, 0.15379584, 0.17032746, 0.22611253, 0.06339709,
       0.16133664, 0.04637148, 0.18075829, 0.19603426, 0.30023079,
       0.12269022, 0.2641298 , 0.10721175, 0.22673048, 0.13451433,
       0.23621555, 0.2753476 , 0.2338715 , 0.29268852, 0.19096995])

In [16]:
top = pd.Series(similarity).sort_values(ascending=False).head(5)
top

9     0.337015
39    0.300231
48    0.292689
19    0.282242
46    0.275348
dtype: float64

In [21]:
for i, score in top.items():
    print(i,score, docs[i][:50])

9 0.3370146678160353 국내 AI반도체 스타트업 리벨리온에 300억원 투자 AI반도체 영역 본격 진입 외산 GPU
39 0.30023079071602865 LG유플러스 LG전자 LG생활건강은 LG그룹 창립 75주년을 기념해 공동 이벤트 함께 걸어
48 0.2926885166059492 LG전자와 SM엔터테인먼트가 홈트레이닝 시장 공략에 나선다 사진은 조주완 LG전자 사장 사
19 0.2822422851825195 신한금융투자 보고서 이데일리 이은정 기자 증시 급락세가 이어진 가운데 2분기 실적시즌이 다
46 0.27534760422798854 슈나이더 일렉트릭이 스마트팩토리 솔루션으로 생산성 향상은 물론 탄소 중립에 앞장선다 탄소 


### 2, 3 단계
- 근거를 프롬프트에 넣기

In [22]:
context = ""

for i in top.index:
    context += docs[i] + "\n\n"

print(context)    


국내 AI반도체 스타트업 리벨리온에 300억원 투자 AI반도체 영역 본격 진입 외산 GPU 의존도 극복 AI반도체 사업 진출 넘어 국내 생태계 조성 국내서 AI 풀스택 확보 초대규모 GPU팜 조성 후 전용 AI반도체 국산화 글로벌 진출 기반 마련 리벨리온 KT 최적의 파트너 KT 리벨리온 AI 반도체 사업 로드맵 KT 제공 파이낸셜뉴스 KT가 리벨리온과 손잡고 국산 AI 반도체를 이용해 초대규모 GPU팜을 구축하는 등 국가 AI 경쟁력을 강화에 나섰다 KT는 6일 리벨리온에 300억원 규모의 전략적 투자를 단행하고 사업 협력에 나선다고 밝혔다 인공지능 AI 반도체 시장은 2030년 1179달러 약 152조1660억원 에 달할 것으로 전망되고 있다 AI원팀으로 외산 의존도 KT는 이번 리벨리온과 협력으로 외국산 AI 반도체 의존도를 줄여 나갈 계획이다 리벨리온은 지난달에도 620억원 규모의 시리즈 A 투자를 유치한 주문형 반도체 ASIC 설계에 특화된 국내 AI 반도체 설계 팹리스 스타트업이다 현재 AI 서비스 개발에 필요한 컴퓨팅 인프라 영역에서 엔비디아 등 외국산 GPU 그래픽 처리 장치 점유율이 80 를 육박하고 있다 이는 지금까지는 대부분의 AI서비스 솔루션이 엔비디아가 제공하는 SW CUDA를 기반으로 개발돼 대부분 AI 반도체 개발사들도 엔비디아 의존도를 떨치기 어려웠다 이에 KT는 AI원팀으로써 협력 중인 스타트업들과 함께 국내 AI풀스택을 구축키로 했다 우선 연내 수천장 규모에 달하는 초대규모 GPU팜을 구축한다 내년에는 해당 GPU팜에 하이퍼스케일 AI컴퓨팅 HAC 전용으로 자체 개발한 AI 반도체를 접목할 예정이다 이 AI 반도체는 AI알고리즘에 최적화된 신경망처리장치 NPU 다 NPU는 GPU 대비 3배 넘는 에너지 효율과 저렴한 도입 비용 복잡한 알고리즘에도 적합한 성능 등이 강점이다 이미 KT는 지난해 kt 클라우드가 출시한 종량제 GPU 서비스 HAC에 CUDA를 지원할 수 있는 자체 AI 프레임워크 적용에 성공했다 이를 기반으로 엔비디

In [23]:
prompt = f"""
아래 [근거 자료]만 참고해서 질문에 답하세요
자료에 없으면 '자료에 없음'이라고 답하세요

[근거자료]
{context}

[질문]
{question}
"""


# 테스트용으로 근거없이 질문해서 답변 받아 봅시다.
response_rag = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {'role':'user','content':prompt}

    ]
)

print(response_rag.choices[0].message.content)



리벨리온은 **국내 AI 반도체 설계 팹리스 스타트업**입니다. 주문형 반도체(ASIC) 설계에 특화되어 있으며, AI 알고리즘에 최적화된 **신경망처리장치(NPU)**를 개발합니다. KT와 협력해 국산 AI 반도체를 개발하고, 초대규모 GPU팜 구축과 국내 AI 반도체 생태계 조성 및 글로벌 진출을 추진하고 있습니다.
